In [ ]:
'''
RFP - 하이브리드 실험

기존 KURE-v1 단독 검색 파이프라인에 BM25 키워드 검색을 결합해, 법률·계약 용어처럼 정확한 단어 매칭이 중요한
질문에서 검색 품질이 개선되는지 확인하는 실험.

핵심 흐름
1. 청크 데이터, KURE 임베딩 인덱스 로드
2. kiwi 형태소 분석기로 전체 청크를 토큰화한 뒤 BM25Okapi로 키워드 기반 인덱스 생성
3. dense(KURE) 유사도와 BM25 점수를 alpha 비율로 결합해 순위를 매기는 핵심 검색 함수
4. 문서 힌트가 없거나 키워드 매칭에 실패했을 때 hybrid_search로 대체 검색하도록 KURE 단독 버전에서 일부 분기만 교체
6. 골든셋 40개에 대해 답변 생성 후 하이브리드 적용 후 평균 점수를 KURE 단독 결과(89.58점)와 비교

결과
- single_doc: 평균 96.67/100
- multi_doc_compare: 평균 77.50/100
- follow_up: 평균 81.67/100
- unknown: 평균 100.00/100

KURE 단독 결과(89.58점) 대비 multi_doc_compare는 소폭 상승(75.00→77.50)했으나
follow_up은 하락(86.67→81.67) - 카테고리별로 결과가 엇갈림.

'''

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install faiss-cpu sentence-transformers openai rank_bm25 kiwipiepy -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 11.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 94.8 MB/s eta 0:00:00


In [3]:
import sys
import types
import pickle
import re
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

src_module = types.ModuleType('src')
chunking_module = types.ModuleType('src.chunking')
class Chunk:
    pass
chunking_module.Chunk = Chunk
src_module.chunking = chunking_module
sys.modules['src'] = src_module
sys.modules['src.chunking'] = chunking_module

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')

with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunk_objects = pickle.load(f)

all_chunks_final = []
chunk_metadata_final = []
for c in chunk_objects:
    all_chunks_final.append(c.text)
    meta_info = c.metadata if c.metadata else {}
    chunk_metadata_final.append({
        '파일명': c.doc_id,
        '발주기관': meta_info.get('발주_기관', ''),
        '사업금액': meta_info.get('사업_금액', None),
        '마감일': meta_info.get('입찰_참여_마감일', ''),
    })
print(f"청크: {len(all_chunks_final)}개")

청크: 18239개


In [ ]:
import torch

with open(DATA_DIR / 'kure_embeddings.pkl', 'rb') as f:
    kure_embeddings = pickle.load(f)

import faiss
index_kure = faiss.IndexFlatL2(kure_embeddings.shape[1])
index_kure.add(np.array(kure_embeddings).astype('float32'))

from sentence_transformers import SentenceTransformer
kure_model = SentenceTransformer('nlpai-lab/KURE-v1', device='cuda', model_kwargs={'torch_dtype': torch.float16})
print(kure_model.device, index_kure.ntotal)

In [6]:
from google.colab import userdata
import openai

api_key = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=api_key)

seen = set()
all_filenames_with_biz = []
for cm in chunk_metadata_final:
    if cm['파일명'] not in seen:
        seen.add(cm['파일명'])
        all_filenames_with_biz.append((cm['파일명'], cm.get('발주기관', '')))
print(f"고유 문서 수: {len(all_filenames_with_biz)}")

고유 문서 수: 98


In [7]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi
from tqdm import tqdm

kiwi = Kiwi()

def tokenize_korean(text):
    return [token.form for token in kiwi.tokenize(text)]

In [8]:
tokenized_chunks = [tokenize_korean(c) for c in tqdm(all_chunks_final)]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 인덱스 생성 완료")

100%|██████████| 18239/18239 [05:36<00:00, 54.24it/s]


BM25 인덱스 생성 완료


In [9]:
ORG_ALIAS_MAP = {
    '대검찰청': ['검찰'],
    '고려대학교': ['고려대'],
    '한국산업단지공단': ['산단'],
}

COMMON_SUFFIX_WORDS = {
    '박물관', '시스템', '센터', '공단', '진흥원', '협회', '재단', '연구원', '공사', '대학교',
    '사업', '관리', '운영', '구축', '개선', '개발', '지원', '정보', '용역', '기관', '기술',
    '고도화', '확대', '기능', '서비스', '일자리', '플랫폼', '통합', '접수',
    '일자리재단', '일자리플랫폼', '보험', '입찰공고', '공고',
    '과학연구', '과학연', '학연구', '연구소', '기록관리', '경기기록',
    '학교', '학교 ', ' 학교', '산학협력단', '산학협력', '학협력단',
    '통합시스템'
}
COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}

LEGAL_KEYWORDS_MAP = {
    '하도급': ['하도급'],
    '공동수급': ['공동수급', '지분율', '컨소시엄'],
    '지분율': ['지분율', '공동수급'],
    '계약보증금': ['계약보증금', '보증금'],
    '평가': ['배점', '평가비율', '기술평가', '가격평가'],
    '제안서 보상': ['제안서 보상'],
    '불이익': ['부정당업자', '입찰보증금', '귀속'],
    '제출물': ['제출서류', '부', 'USB', '제출규격'],
    '제출': ['제출서류', 'USB'],
    '수량': ['부', 'USB'],
    '구축기간': ['사업기간', '구축기간', '개월'],
    '사업기간': ['사업기간', '구축기간', '개월'],
    '유지보수': ['무상유지보수', '유지보수기간', '하자보수', '무상 하자보수'],
    '참가자격': ['참가자격', '참가 자격'],
    '유지관리': ['하자보수', '유지관리 인력', '무상 하자보수'],
    '교육 의무': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '교육을': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '검수 후': ['하자보수', '유지관리 인력'],
    '재입찰': ['재입찰', '재공고입찰', '최초의 입찰'],
    '재공고': ['재입찰', '재공고입찰', '최초의 입찰'],
    '조건 변경': ['재입찰', '재공고입찰', '최초의 입찰'],
    '지역 요건': ['주된 영업소', '소재지'],
    '부산에': ['주된 영업소', '소재지'],
    '지역요건': ['주된 영업소', '소재지'],
    '소재지': ['주된 영업소', '소재지'],
    '보유인력': ['보유인력', '배점한도'],
    '배점한도': ['보유인력', '배점한도'],
    '계량평가': ['보유인력', '배점한도', '재무구조'],
    '규모비율': ['규모비율', '환산점수', '점수비중'],
    '환산점수': ['규모비율', '환산점수', '점수비중'],
    '수행실적': ['규모비율', '환산점수', '수행실적'],
    '신인도': ['신인도', '가점'],
    '가점표': ['신인도', '가점'],
    '연구원 승인': ['Lesson', '회람'],
    '발생한 경우': ['Lesson', '회람'],
    '회람': ['Lesson', '회람'],
}

CORRECTED_TABLES = {
    ('서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf', '약자기업'):
        "[신인도 가점표 - 약자기업 지원 및 정책적 지원 항목별 점수 (정확히 매칭됨)]\n"
        "1. 장애인 기업(중소벤처기업부 발급) : 1점\n2. 여성기업(중소벤처기업부 발급) : 1점\n"
        "3. 중증장애인생산품 생산시설(보건복지부 지정) : 1점\n4. 사회적 기업(고용노동부 지정) : 1점\n"
        "5. 예비 사회적 기업(지방자치단체 지정) : 1점\n6. 사회적협동조합(정부부처 지정) : 1점\n"
        "7. 자활기업(지방자치단체 지정) : 1점\n8. 가족친화 우수기업 : 0.8점\n9. 하도급거래 모범기업 : 0.8점\n"
        "10. 노사문화 우수기업 : 0.5점\n11. 남녀고용평등 우수기업 : 0.5점\n12. 모범납세자 : 0.3점"
}

In [10]:
# 조건 필터링 + KURE 벡터 검색 + BM25/KURE 하이브리드 검색
# hybrid_search: dense(KURE) 점수와 BM25 점수를 alpha 비율로 결합해 순위를 매김
# dense_scores: L2 거리를 1-정규화해 유사도 점수로 변환
# bm25_scores: 최댓값으로 정규화
# combined_scores = alpha * dense + (1-alpha) * bm25
# 문서당 최대 3개 청크까지만 포함해 결과 다양성 확보

def find_relevant_keywords(question):
    matched = []
    for trigger, kws in LEGAL_KEYWORDS_MAP.items():
        if trigger in question:
            matched.extend(kws)
    return list(set(matched))


def is_aggregation_question(question):
    keywords = ['몇 개', '개수', '다 나열', '몇 건']
    strong_total = '전부' in question or ('총' in question and ('개' in question or '건' in question))
    return any(kw in question for kw in keywords) or strong_total


def extract_filter_conditions(query):
    conditions = {}
    if '억' in query and ('이상' in query or '넘는' in query):
        match = re.search(r'(\d+)억', query)
        if match:
            conditions['금액_최소'] = int(match.group(1)) * 100000000
    if '지자체' in query or '지방자치단체' in query:
        conditions['지자체'] = True
    if '공사' in query and ('OO공사' in query or '발주기관이' in query):
        conditions['공사'] = True
    if 'AI' in query:
        conditions['주제_AI'] = True
    if '긴급' in query:
        conditions['긴급'] = True
    if '보안' in query:
        conditions['보안'] = True
    if '재난' in query:
        conditions['재난'] = True
    return conditions


def is_local_gov(org):
    if org is None or (isinstance(org, float)):
        return False
    return bool(re.search(r'(광역시|특별시|특별자치도|특별자치시|[가-힣]+도|[가-힣]+시|[가-힣]+군|[가-힣]+구)$', str(org).strip()))


def get_filtered_candidates(conditions, chunk_metadata):
    if not conditions:
        return None
    doc_info = {}
    for cm in chunk_metadata:
        fname = cm['파일명']
        if fname not in doc_info:
            doc_info[fname] = cm
    allowed = set()
    for fname, info in doc_info.items():
        ok = True
        if '금액_최소' in conditions:
            amt = info.get('사업금액')
            if amt is None or amt < conditions['금액_최소']:
                ok = False
        if conditions.get('지자체'):
            if not is_local_gov(info.get('발주기관')):
                ok = False
        if conditions.get('공사'):
            org = str(info.get('발주기관', ''))
            if '공사' not in org:
                ok = False
        if conditions.get('긴급'):
            if '긴급' not in fname:
                ok = False
        if conditions.get('재난'):
            if '재난' not in fname:
                ok = False
        if ok:
            allowed.add(fname)
    return allowed if allowed else None


def search_with_filter(query, index, model, chunk_metadata, all_chunks, k=10, max_per_doc=1):
    conditions = extract_filter_conditions(query)
    allowed_filenames = get_filtered_candidates(conditions, chunk_metadata)
    query_embedding = model.encode([query])
    search_k = min(len(all_chunks), 2000)
    distances, indices = index.search(np.array(query_embedding).astype('float32'), search_k)
    seen_docs = {}
    results = []
    for i in indices[0]:
        doc_name = chunk_metadata[i]['파일명']
        if allowed_filenames is not None and doc_name not in allowed_filenames:
            continue
        count = seen_docs.get(doc_name, 0)
        if count < max_per_doc:
            results.append(i)
            seen_docs[doc_name] = count + 1
        if len(results) >= k:
            break
    return results


def normalize_org_name(name):
    return re.sub(r'(특별시|광역시|특별자치시|특별자치도)', '', name)


def hybrid_search(query, top_k_candidates=1500, final_k=10, alpha=0.5):
    query_embedding = kure_model.encode([query])
    dense_distances, dense_indices = index_kure.search(np.array(query_embedding).astype('float32'), top_k_candidates)
    max_dist = dense_distances[0].max()
    dense_scores = {idx: 1 - (dist / max_dist) for idx, dist in zip(dense_indices[0], dense_distances[0])}
    tokenized_query = tokenize_korean(query)
    bm25_scores_all = bm25.get_scores(tokenized_query)
    max_bm25 = bm25_scores_all.max() if bm25_scores_all.max() > 0 else 1
    combined_scores = {}
    for idx in dense_indices[0]:
        dense_s = dense_scores.get(idx, 0)
        bm25_s = bm25_scores_all[idx] / max_bm25
        combined_scores[idx] = alpha * dense_s + (1 - alpha) * bm25_s
    ranked = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
    results = []
    seen_docs = {}
    for idx, score in ranked:
        doc_name = chunk_metadata_final[idx]['파일명']
        count = seen_docs.get(doc_name, 0)
        if count < 3:
            results.append(idx)
            seen_docs[doc_name] = count + 1
        if len(results) >= final_k:
            break
    return results

In [11]:
def extract_doc_hints_multi(question, all_filenames_with_biz):
    q_no_space = question.replace(' ', '').replace('&', '')
    org_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        org_part = fname.replace('refined_', '').split('_')[0].strip()
        org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
        org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
        org_core_clean = re.sub(r'\s*입찰공고\s*$', '', org_core_clean).strip()
        org_core_norm = normalize_org_name(org_core_clean)
        if len(org_core_clean) < 2:
            continue
        matched = False
        if org_core_clean in question:
            matched = True
        elif len(org_core_norm) >= 3 and org_core_norm in question:
            matched = True
        elif org_core_clean in ORG_ALIAS_MAP and any(alias in question for alias in ORG_ALIAS_MAP[org_core_clean]):
            matched = True
        else:
            min_len = 4
            for target_str in [org_core_clean, org_core_norm]:
                for start in range(len(target_str) - min_len + 1):
                    for length in range(len(target_str) - start, min_len - 1, -1):
                        substr = target_str[start:start+length]
                        if substr.strip() in question and substr.strip() not in COMMON_SUFFIX_WORDS:
                            matched = True
                            break
                    if matched:
                        break
                if matched:
                    break
        if matched:
            org_candidates.append((fname, org_core_clean))

    biz_candidates = []
    quoted = re.findall(r"['\"]([^'\"]+)['\"]", question)
    for fname, biz_name in all_filenames_with_biz:
        biz_name = str(biz_name).strip()
        if len(biz_name) >= 4 and biz_name in question:
            biz_candidates.append(fname)
            continue
        for q in quoted:
            if q in biz_name or biz_name in q:
                biz_candidates.append(fname)
                break
        eng_words = re.findall(r'[A-Za-z][A-Za-z&\s]{2,}[A-Za-z]', biz_name)
        for ew in eng_words:
            ew_no_space = ew.strip().replace(' ', '').replace('&', '')
            if len(ew_no_space) >= 4 and ew_no_space in q_no_space:
                biz_candidates.append(fname)
                break

    stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
    raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', question) if len(w) >= 4]
    keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]

    def fuzzy_match(kw, text, min_overlap=4):
        kw_ns = kw.replace(' ', '')
        text_ns = text.replace(' ', '')
        if kw_ns in text_ns:
            return True
        for n in range(len(kw_ns), min_overlap - 1, -1):
            if kw_ns[:n] in text_ns:
                return True
        return False

    def keyword_weight(kw):
        return 3 if re.search(r'[A-Za-z]', kw) else 1

    filename_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        fname_clean = fname.replace('refined_', '').replace('.hwp', '').replace('.pdf', '')
        matched_kws = [kw for kw in keywords_all if fuzzy_match(kw, fname_clean)]
        score = sum(keyword_weight(kw) for kw in matched_kws)
        if score > 0:
            filename_candidates.append((fname, score, len(matched_kws)))

    if filename_candidates:
        filename_candidates.sort(key=lambda x: -x[1])
        max_score = filename_candidates[0][1]
        for top_fname, score, cnt in filename_candidates:
            if score >= max_score * 0.6 or score >= 1:
                if top_fname not in [f for f, _ in org_candidates] and top_fname not in biz_candidates:
                    if len(filename_candidates) <= 3 or score >= max(max_score * 0.6, 1):
                        biz_candidates.append(top_fname)

    org_groups = {}
    for fname, org_core in org_candidates:
        org_groups.setdefault(org_core, []).append(fname)

    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
    keywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords]

    final_hints = []
    for org_core, fnames in org_groups.items():
        fnames = list(set(fnames))
        if len(fnames) == 1:
            final_hints.append(fnames[0])
        else:
            fname_to_biz = dict(all_filenames_with_biz)
            best_doc, best_score2 = None, -1
            for fname in fnames:
                biz_name = fname_to_biz.get(fname, '')
                score2 = sum(1 for kw in keywords if kw in fname or kw in str(biz_name))
                if score2 > best_score2:
                    best_score2, best_doc = score2, fname
            final_hints.append(best_doc)

    for fname in biz_candidates:
        if fname not in final_hints:
            final_hints.append(fname)

    return list(dict.fromkeys(final_hints))

In [12]:
SYSTEM_PROMPT_NEW_V2 = """
너는 'RFP 챗봇'이야. 입찰메이트 컨설턴트가 제안요청서(RFP) 문서를 빠르게 파악할 수 있게 도와줘.

## 기본 원칙

1. 반드시 아래에 제공된 문서 내용(컨텍스트)에 근거해서만 답변해. 문서에 없는 내용을 추측하거나 지어내지 마.

2. 답변은 간결하고 명확하게 작성해. 불필요한 서론 없이 핵심부터 답해.

3. 질문 유형에 따라 답변 형식을 다르게 해:
   - 단일 사실 조회 (예: "예산이 얼마야?") → 핵심 수치/사실 위주로 간결하게
   - 두 개 이상 비교 (예: "A랑 B 중 뭐가 더 커?") → 각 항목을 나란히 제시하고 비교 결론 제시
   - 목적/배경을 묻는 질문 → 관련 섹션을 요약해서 설명
   - 조건에 맞는 여러 문서를 찾는 질문 → 목록 형태로 정리

4. 이전 대화에서 언급된 문서나 주제가 있으면, 후속 질문("그럼 마감일은?" 등)은 같은 문서/주제 맥락에서 답변해.

5. 답변 끝에는 근거가 된 문서명을 명시해.

## 답변을 거절/기권해야 하는 경우 (매우 중요)

아래 경우에는 문서 안에서 관련 정보를 억지로 찾아서 답하려 하지 말고, 명확히 "답변할 수 없다"고만 말하고 끝내. 관련 있어 보이는 부가 정보를 나열하지 마.

- **범위 밖 요청(out_of_scope)**: 네가 할 수 없는 행동을 요청하는 경우(전화 걸기, 이메일 보내기, 실시간 조회 등), 또는 "오늘", "지금", "최신"처럼 실시간·최신 정보를 요구하는 경우. 이때는 "이 기능은 제가 수행할 수 없습니다" 또는 "실시간 정보는 제공된 문서에서 확인할 수 없습니다"라고만 답하고, 대신 관련 문서를 찾아주거나 연락처를 나열하는 등 다른 시도를 하지 마.

- **근거 부족(insufficient_evidence)**: 낙찰 결과, 경쟁사 현황, 예상 낙찰가처럼 애초에 이 문서(제안요청서)에 있을 수 없는 정보를 물어보는 경우. "확인되지 않습니다"라고만 답해.

- **판단/추측 요청(ambiguous)**: "우리 회사가 자격을 충족하는지 판정해줘", "수주 확률이 얼마냐" 처럼 사용자의 상황과 문서를 대조해서 네가 주관적으로 판단·확률을 계산해야 하는 질문. 이런 판정이나 확률 계산은 네가 할 수 없다고 답하고, 판단에 필요한 조건 목록만 간단히 안내해도 되지만 장황하게 체크리스트를 만들지는 마.

- **사용자가 임의의 가정을 세우고 그 가정으로 확정 답변을 요구하는 경우**: "문서에 없으면 OO라고 가정하고 확정해줘"처럼, 사용자가 제시한 임의의 규칙(추측)을 근거 삼아 사실인 것처럼 답을 만들어달라는 요청. 이건 절대 받아들이지 마. "문서에 없는 정보는 임의로 가정해서 확정할 수 없습니다"라고 답하고, 사용자가 제안한 가정을 그대로 적용해서 계산해주지 마.

## 표 형식 데이터 안내

컨텍스트에 [표]라는 표시와 함께 "항목 | 값" 형태로 된 부분이 나오면, 이는 원본 문서의 표를 옮긴 것이야. 각 줄은 표의 한 행을 의미하고, |로 구분된 각 항목은 표의 열(칸)을 의미해. 이 형식을 참고해서 항목과 값을 정확히 짝지어 답변해.

## 여러 문서 처리

컨텍스트에 여러 문서의 내용이 섞여 있을 수 있어. 각 문서 조각이 어느 문서(파일명)에서 왔는지 구분해서, 서로 다른 문서의 정보를 혼동하거나 섞어서 답하지 마.

일부 정보(예: 긴급 여부, 재공고 여부)는 본문 내용이 아니라 문서명(파일명)에만 표시되어 있을 수 있어. 문서명에 이런 정보가 있으면 그것도 근거로 활용해서 답해.

## 주제/카테고리 판단 시 주의사항

질문의 키워드와 문서 안의 유사한 단어가 겉보기에 비슷해 보여도, 실제 의미는 다를 수 있어. 문서의 실제 사업 목적과 내용까지 확인해서 질문 의도와 정확히 일치하는지 판단하고, 확신이 안 서면 "이 문서는 [실제 의미]를 다루고 있어 질문 의도와 다를 수 있습니다"처럼 구분해서 답해. 단어의 표면적 유사성만으로 포함시키지 마.

아래는 실제로 혼동이 발생했던 사례야. 반드시 참고해서 판단해:

예시: "재난 관련 사업을 찾아줘"라는 질문에, 사업명이 "적십자병원 병원정보 재해복구시스템 구축 용역"인 문서가 검색됐다고 하자. 이 문서는 재난(자연재해, 재난관리) 관련 사업이 아니야. "재해복구시스템(Disaster Recovery System)"은 서버/데이터베이스 장애 시 데이터를 복구하는 IT 인프라 용어이고, "병원정보시스템 데이터베이스 운영"을 다루는 순수 IT 시스템 구축 사업이야. 이 사업을 "재난 관련"으로 포함시키면 틀린 답변이야. 반드시 제외해.

마찬가지로 "응급의료 상황관리시스템"(병원 전원·환자 이송을 지원하는 IT 시스템)도 "재난 관리 시스템"과는 다른 목적의 사업이야. 재난은 지진, 홍수, 화재 등 자연재해나 사회재난에 대응하는 시스템을 뜻하며, 단순히 "응급", "긴급", "재해" 같은 단어가 사업명에 있다고 해서 재난 관련 사업으로 분류하면 안 돼.

## 질문 해석 관련

질문에 "OO", "XX" 같은 placeholder처럼 보이는 표현이 있어도, 이는 실제로 채워야 할 빈칸이 아니라 "특정 패턴을 가진 이름 전체"를 가리키는 일반적인 화법일 수 있어. 예를 들어 "발주기관이 OO공사인 사업"은 "발주기관명이 '공사'로 끝나는 모든 사업"을 뜻하는 것이지, 사용자가 실제 공사명을 지정해줘야 한다는 뜻이 아니야. 이런 경우 되묻지 말고, 컨텍스트 안에서 해당 패턴에 맞는 사업을 최대한 찾아서 답해.

## 금액 표기 관련

금액은 부가세(VAT) 포함/별도 표기가 문서마다 다를 수 있어. 답변할 때 원문에 표기된 형태(포함/별도 여부 포함) 그대로 전달하고, 임의로 환산하지 마.

## 구조화된 필드(공고번호, 사업금액, 입찰 참여 시작일/마감일, 발주기관) 답변 규칙

이 필드들은 컨설턴트의 실제 입찰 결정에 직결되니까 특히 신중하게 답해.

- 검색된 문서 조각과 메타데이터에 명확한 값이 있으면, 근거와 함께 답변해.
- 값이 없거나 불확실하면 절대 추정하지 말고 "확인되지 않습니다"라고 명확히 답해.
- 아래 함정에 특히 주의해:
  - 공고번호를 유사한 다른 번호나 제목의 "[재공고]" 표시만으로 추정하지 마.
  - 개찰 시각이나 제안서 평가 시각을 입찰 참여 마감일로 착각해서 답하지 마. 이 셋은 서로 다른 시점이야.
  - 공개일(공고가 게시된 날짜)을 입찰 참여 시작일로 대체하지 마.
  - 발주기관은 게시기관·수요기관·계약기관이 다를 수 있으니까, 근거 없이 하나를 임의로 선택하지 마.
  - 사업금액이 0원이나 1원으로 보이면, 이건 실제 금액이 아니라 비공개·미확정을 나타내는 표시일 수 있어. 이 경우 실금액처럼 답하지 말고 "금액이 비공개이거나 미확정 상태로 보입니다"라고 답해.

## 참가자격 / 제한조건 / 평가기준 / 제출요건 / 계약 리스크(위약금, 계약보증금 등) 답변 규칙

이 항목들도 컨설턴트가 실제로 입찰 여부를 판단하고 계약 의무를 이해하는 데 직결되니까 신중하게 답해.

- 검색된 문서 조각 안에 명확한 근거가 있을 때만 답변해.
- 명확한 근거가 없으면 "제공된 문서 범위에서는 확인되지 않습니다. 원문 전체 확인이 필요할 수 있습니다"라고 답해.
- 다른 사업의 일반적인 조항이나 통상적인 관행을 이 사업에 적용해서 답하지 마.

## 부분 정보 처리

질문에 여러 정보가 섞여 있고 그중 일부만 확인 가능하면, 확인되는 정보는 근거와 함께 답하고 확인 안 되는 정보만 위 규칙에 따라 "확인되지 않습니다"라고 답해. 일부가 확인 안 된다고 전체 답변을 포기하지 마.

## 컨텍스트 (검색된 문서 조각)
{context}

## 질문
{question}
"""

In [13]:
# 답변 생성 함수
# 벡터 검색만으로 처리하던 두 지점을 hybrid_search로 교체함
# 표 보정(CORRECTED_TABLES) 최우선 적용
# 집계형 질문 → 관련도 높은 문서 하나의 청크 전체 사용
# 문서 힌트 1개 + 키워드 있음 → 키워드 매칭 청크, 없으면 hybrid_search로 대체
# 문서 힌트 2개 이상 → 비교 질문, 각 문서 청크 일부씩
# 문서 힌트 1개만 → 해당 문서 앞쪽 청크 15개
# 조건 필터만 있음 → 필터 통과 문서군 내 KURE 단독 검색
# 아무 힌트도 없음 → hybrid_search로 전체 대상 검색

def ask_rfp_final(question, model_name="gpt-5-mini", max_retries=2):
    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]
    keywords = find_relevant_keywords(question)
    conditions = extract_filter_conditions(question)
    allowed = get_filtered_candidates(conditions, chunk_metadata_final)

    fname_to_meta = {}
    for cm in chunk_metadata_final:
        if cm['파일명'] not in fname_to_meta:
            fname_to_meta[cm['파일명']] = cm

    def meta_header(fname):
        m = fname_to_meta.get(fname, {})
        org = m.get('발주기관', '')
        amt = m.get('사업금액')
        amt_str = f"{amt:,.0f}원" if amt not in (None, '') else "확인되지 않음"
        return f"[문서: {fname}]\n[발주기관(메타데이터): {org}]\n[사업금액(메타데이터): {amt_str}]"

    context_parts = []

    for (fname_key, kw_key), corrected_text in CORRECTED_TABLES.items():
        if fname_key in doc_hints and kw_key in question:
            context_parts.append(f"[문서: {fname_key}]\n{corrected_text}")

    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0], -1
        for fname in doc_hints:
            biz = fname_to_meta.get(fname, {}).get('발주기관', '')
            score = sum(1 for kw in qkeywords if kw in fname or kw in str(biz))
            if score > best_score:
                best_score, best_doc = score, fname
        doc_hint = best_doc
        doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
        header = meta_header(doc_hint)
        for i in doc_chunks:
            context_parts.append(f"{header}\n{all_chunks_final[i]}")

    elif len(doc_hints) == 1 and keywords:
        doc_hint = doc_hints[0]
        doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
        keyword_chunks = [i for i in doc_chunks if any(kw in all_chunks_final[i] for kw in keywords)]
        result_indices = keyword_chunks[:25] if keyword_chunks else hybrid_search(question, final_k=10)
        for i in result_indices:
            fname = chunk_metadata_final[i]['파일명']
            context_parts.append(f"{meta_header(fname)}\n{all_chunks_final[i]}")

    elif len(doc_hints) >= 2:
        for doc_hint in doc_hints:
            doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
            if keywords:
                matched = [i for i in doc_chunks if any(kw in all_chunks_final[i] for kw in keywords)]
                selected = matched[:8] if matched else doc_chunks[:8]
            else:
                selected = doc_chunks[:8]
            header = meta_header(doc_hint)
            for i in selected:
                context_parts.append(f"{header}\n{all_chunks_final[i]}")
    elif doc_hints:
        doc_hint = doc_hints[0]
        doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
        header = meta_header(doc_hint)
        for i in doc_chunks[:15]:
            context_parts.append(f"{header}\n{all_chunks_final[i]}")
    elif allowed:
        k = min(len(allowed), 80)
        result_indices = search_with_filter(question, index_kure, kure_model, chunk_metadata_final, all_chunks_final, k=k, max_per_doc=1)
        for i in result_indices:
            fname = chunk_metadata_final[i]['파일명']
            context_parts.append(f"{meta_header(fname)}\n{all_chunks_final[i]}")
    else:
        result_indices = hybrid_search(question, final_k=10)
        for i in result_indices:
            fname = chunk_metadata_final[i]['파일명']
            context_parts.append(f"{meta_header(fname)}\n{all_chunks_final[i]}")

    context = "\n\n---\n\n".join(context_parts)
    final_prompt = SYSTEM_PROMPT_NEW_V2.format(context=context, question=question)

    for attempt in range(max_retries):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": final_prompt}],
            max_completion_tokens=8000,
            reasoning_effort="low"
        )
        answer = response.choices[0].message.content
        if answer:
            return answer
    return "(답변 생성 실패)"

In [16]:
with open(DATA_DIR / 'dev.refined.review-candidate.jsonl', encoding='utf-8') as f:
    golden40 = [json.loads(l) for l in f if l.strip()]

hybrid_answers = []
for item in golden40:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} (이 문서 기준) {question}"
        answer = ask_rfp_final(combined_q)
    else:
        answer = ask_rfp_final(question)

    hybrid_answers.append({'case_id': cid, 'task_type': task_type, 'question': question, 'answer': answer})
    print(f"[{cid}][{task_type}] {question}")
    print(answer)
    print()

[dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 243,000,000원 — 부가가치세(VAT) 포함. 근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp

[dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
- 수행기간: 계약일로부터 120일(약 4개월)  
- 사업예산: 70,000,000원(금칠천만원, VAT 포함)

근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp

[dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 차수: 2차 (1차 사업: 계약일로부터 4개월, 2차 사업: 1차 완료일로부터 2개월)  
- 평가비중: 기술평가 90% / 가격평가 10%

근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp

[dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
- 사업기간: 계약일로부터 90일(문서상 별도 표기: 계약일로부터 3개월, 2024년 11월 1일까지).  
- 시범 도입 규모: 1단계 시범도입 3개 기관(서울 2개소, 울산 1개소). 하드웨어 수량 표는 3개소 기준(예: RFID 고정형 리더기 3대, 이동형 리더기 3대, 안테나 12개, RFID태그 2,400개 등).  

근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입 제안요청서 (도.hwp)

[dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출방법: 입찰서와 제안서는 전자적으로 나라장터(e-발주시스템, http:/

In [17]:
def normalize_text(t):
    return t.replace(',', '').replace(' ', '')


def normalize_dates(text):
    text = re.sub(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', r'\1.\2.\3', text)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', lambda m: f"{m.group(1)}.{int(m.group(2)):02d}.{int(m.group(3)):02d}", text)
    return text


def check_fact_included(fact_text, answer_text):
    fact_text = normalize_dates(fact_text)
    answer_text_n = normalize_dates(answer_text)
    answer_norm = normalize_text(answer_text_n)

    numbers = re.findall(r'\d+(?:\.\d+)?', fact_text)
    numbers = [n for n in numbers if len(n) >= 2]
    for num in numbers:
        if num in answer_norm:
            continue
        if len(num) == 4 and num.startswith('20'):
            if num[2:] in answer_norm:
                continue
        num_no_zero = re.sub(r'^0+', '', num)
        if num_no_zero and num_no_zero in answer_norm:
            continue
        return False

    raw_words = re.split(r'[\s,·:()]+', fact_text)
    stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
    core_words = []
    for w in raw_words:
        w = w.rstrip('.,')
        if len(w) < 2:
            continue
        if re.match(r'^\d', w):
            continue
        for suf in stopwords_suffix:
            if w.endswith(suf) and len(w) > len(suf):
                w = w[:-len(suf)]
                break
        w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
        if len(w) >= 2:
            core_words.append(w)

    if not core_words:
        return True

    match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
    return match_count / max(len(core_words), 1) >= 0.4


def official_score(item, answer_text):
    key_points = item['gold'].get('required_key_points', [])
    decision = item['gold'].get('decision')

    if decision == 'abstain':
        abstain_phrases = ['확인되지 않습니다', '답변할 수 없', '수행할 수 없', '확인할 수 없', '판단할 수 없',
                            '판정할 수 없', '판정해줄 수 없', '계산할 수 없', '제공할 수 없', '받아들일 수 없',
                            '확정할 수 없', '알려드릴 수 없', '불가능합니다', '제공된 문서 범위에서는']
        is_abstained = any(p in answer_text for p in abstain_phrases)
        return 100 if is_abstained else 0

    if not key_points:
        return None

    included = [check_fact_included(kp['text'], answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

In [18]:
scored_results = []
for item in golden40:
    hy = next(h for h in hybrid_answers if h['case_id'] == item['case_id'])
    score = official_score(item, hy['answer'])
    scored_results.append({'case_id': item['case_id'], 'task_type': item['task_type'], 'score': score})
    print(f"[{item['case_id']}][{item['task_type']}] 점수: {score}")

valid = [r['score'] for r in scored_results if r['score'] is not None]
print(f"\n 하이브리드 적용 후 전체 평균: {sum(valid)/len(valid):.2f}/100")

by_type = {}
for r in scored_results:
    if r['score'] is not None:
        by_type.setdefault(r['task_type'], []).append(r['score'])
for t, scores in by_type.items():
    print(f"{t}: 평균 {sum(scores)/len(scores):.2f}/100")

[dev-single-001][single_doc] 점수: 100.0
[dev-single-002][single_doc] 점수: 100.0
[dev-single-003][single_doc] 점수: 100.0
[dev-single-004][single_doc] 점수: 100.0
[dev-single-005][single_doc] 점수: 100.0
[dev-single-006][single_doc] 점수: 100.0
[dev-single-007][single_doc] 점수: 100.0
[dev-single-008][single_doc] 점수: 100.0
[dev-single-009][single_doc] 점수: 100.0
[dev-single-010][single_doc] 점수: 66.67
[dev-multi-001][multi_doc_compare] 점수: 100.0
[dev-multi-002][multi_doc_compare] 점수: 66.67
[dev-multi-003][multi_doc_compare] 점수: 66.67
[dev-multi-004][multi_doc_compare] 점수: 66.67
[dev-multi-005][multi_doc_compare] 점수: 50.0
[dev-multi-006][multi_doc_compare] 점수: 100.0
[dev-multi-007][multi_doc_compare] 점수: 75.0
[dev-multi-008][multi_doc_compare] 점수: 100.0
[dev-multi-009][multi_doc_compare] 점수: 100.0
[dev-multi-010][multi_doc_compare] 점수: 50.0
[dev-followup-001][follow_up] 점수: 100.0
[dev-followup-002][follow_up] 점수: 50.0
[dev-followup-003][follow_up] 점수: 100.0
[dev-followup-004][follow_up] 점수: 100.0
[dev